In [2]:
import os
from pathlib import Path
import pandas as pd
import numpy as np


cwd = Path.cwd()
print(cwd)
root=cwd.parents[1]
# os.chdir(root/"Seb_branch"/"institutional-roi-analysis"/"notebooks")
pd.set_option("display.max_columns",None)
display(root)

c:\Users\sebas\PycharmProjects\Git\Seb_branch\institutional-roi-analysis\notebooks\national_models


WindowsPath('c:/Users/sebas/PycharmProjects/Git/Seb_branch/institutional-roi-analysis')

### Feature Selection for Explanatory Model

Variables used in the initial prediction model (e.g., credential level, distance, and other structural constraints) were excluded from the explanatory model.

This is because the residuals already represent performance after controlling for these factors. Including them again would introduce circular reasoning and reduce the interpretability of the results.

Instead, the explanatory model focuses on institutional characteristics and program composition variables that were not used in the prediction stage, allowing us to better understand what drives over- and underperformance.

In [44]:
driver_df=pd.read_csv(root/'data'/'raw'/'scorecard'/'raw_national_inst_driver.csv')

In [45]:
driver_df["has_endowment"] = (
    driver_df["endowment_begin"].notna() &
    driver_df["endowment_end"].notna()
).astype(int)

In [46]:
driver_df["has_endowment"].describe()

count    6322.000000
mean        0.422018
std         0.493920
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max         1.000000
Name: has_endowment, dtype: float64

In [47]:
driver_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6322 entries, 0 to 6321
Data columns (total 49 columns):
 #   Column                                                   Non-Null Count  Dtype  
---  ------                                                   --------------  -----  
 0   program_percentage_agriculture                           5582 non-null   float64
 1   program_percentage_resources                             5582 non-null   float64
 2   program_percentage_architecture                          5582 non-null   float64
 3   program_percentage_ethnic_cultural_gender                5582 non-null   float64
 4   program_percentage_communication                         5582 non-null   float64
 5   program_percentage_communications_technology             5582 non-null   float64
 6   program_percentage_computer                              5582 non-null   float64
 7   program_percentage_personal_culinary                     5582 non-null   float64
 8   program_percentage_education

In [51]:
xgb_residual_df = pd.read_csv(root/"data"/"raw"/"scorecard"/"raw_xgb_national_residual_programs.csv")

In [52]:
xgb_residual_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36219 entries, 0 to 36218
Data columns (total 58 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   code                            36219 non-null  int64  
 1   credential_level                36219 non-null  int64  
 2   unit_id                         36219 non-null  int64  
 3   distance                        36219 non-null  int64  
 4   school_type                     36219 non-null  object 
 5   5_yr_working_count              36219 non-null  float64
 6   school_state                    36219 non-null  object 
 7   location_lat                    36219 non-null  float64
 8   location_lon                    36219 non-null  float64
 9   locale                          36219 non-null  int64  
 10  carnegie_size_setting           36219 non-null  object 
 11  admission_rate_overall          25209 non-null  float64
 12  median_family_income            

In [ ]:
xgb_residual_df['sign_agreement'] = (
    np.sign(xgb_residual_df['1_year_error']) == 
    np.sign(xgb_residual_df['4_year_error'])
) & (
    np.sign(xgb_residual_df['4_year_error']) == 
    np.sign(xgb_residual_df['5_year_error'])
)

print(xgb_residual_df['sign_agreement'].value_counts(normalize=True))

print(xgb_residual_df[['1_year_error','4_year_error','5_year_error']].corr())

sign_agreement
True     0.517325
False    0.482675
Name: proportion, dtype: float64
              1_year_error  4_year_error  5_year_error
1_year_error      1.000000      0.691640      0.572742
4_year_error      0.691640      1.000000      0.665778
5_year_error      0.572742      0.665778      1.000000


In [70]:
xgb_residual_df['total_pred'] = xgb_residual_df[
    ["1_year_pred", "4_year_pred", "5_year_pred"]
].sum(axis=1)

xgb_residual_df["total_count"] = (
    xgb_residual_df["1_yr_working_count"] +
    xgb_residual_df["4_yr_working_count"] +
    xgb_residual_df["5_yr_working_count"]
)

k = np.percentile(np.log1p(xgb_residual_df["total_count"]), 75)

xgb_residual_df["weight"] = (
    np.log1p(xgb_residual_df["total_count"]) /
    np.log1p(xgb_residual_df["total_count"] + k)
)

xgb_residual_df["combined_pct_error"] = xgb_residual_df[
    ["1_year_error", "4_year_error", "5_year_error"]
].median(axis=1) / xgb_residual_df['total_pred'] * 3

# consistent_mask = xgb_residual_df['sign_agreement'] == True
# xgb_residual_df = xgb_residual_df[consistent_mask]
# print(f"Consistent schools: {consistent_mask.sum()} of {len(xgb_residual_df)}")

xgb_residual_df.head()

,code,credential_level,unit_id,distance,school_type,5_yr_working_count,school_state,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students,selectivity_bucket,5_year_earning_log,5_year_pred_log,title,school_name,5_year_earning,5_year_pred,5_year_error,4_year_earning,4_year_earning_log,4_year_pred,4_year_pred_log,4_year_error,4_yr_working_count,1_year_earning,1_year_earning_log,1_year_pred,1_year_pred_log,1_year_error,1_yr_working_count,school_count,confidence,1_year_pct_error,1_year_score,4_year_pct_error,4_year_score,5_year_pct_error,5_year_score,rank_1,rank_4,rank_5,move_1_to_4,move_4_to_5,move_1_to_5,rank_std,rank_std_pct,mean_rank_std_pct,median_score,total_pred,total_count,weight,combined_pct_error,dollar_error_median,dollar_pred_median,row_pct_error,sign_agreement
0,100,3,110422,1,Public,21.0,CA,35.299513,-120.657311,23,16.0,0.3132,74513.0,0.455561,2.0,20.0,1,NaN,NaN,1036.0,mid,11.132397,11.212030,"Agriculture, General.",California Polytechnic State University-San Lu...,68350.0,74015.540,-5665.539062,84412.0,11.343465,72705.680,11.194175,11706.320313,36,64786.0,11.078845,45640.770,10.728557,19145.230469,18.0,28,high,0.419477,0.389775,0.161010,0.155674,-0.076545,-0.072040,1.0,2.0,14.0,1.0,12.0,13.0,7.234178,0.267933,0.183302,0.155674,192361.990,75.0,0.983097,0.182567,11706.320313,72705.680,0.161010,False
1,100,3,130934,1,Public,24.0,DE,39.187173,-75.540530,13,14.0,0.4658,39554.0,0.685488,2.0,20.0,1,940.0,NaN,746.0,mid,10.768022,10.985131,"Agriculture, General.",Delaware State University,47478.0,58990.477,-11512.476562,52676.0,10.871915,58480.465,10.976448,-5804.464844,24,38873.0,10.568055,40523.883,10.609647,-1650.882813,22.0,28,high,-0.040739,-0.038425,-0.099255,-0.094049,-0.195158,-0.185260,18.0,16.0,24.0,-2.0,8.0,6.0,4.163332,0.154197,0.183302,-0.094049,157994.825,70.0,0.981691,-0.110215,-5804.464844,58480.465,-0.099255,True
2,100,3,145813,1,Public,132.0,IL,40.509403,-88.990058,22,16.0,0.8815,67099.0,0.481270,2.0,20.0,1,1113.0,24.0,2439.0,open,11.067279,10.990893,"Agriculture, General.",Illinois State University,64041.0,59331.367,4709.632813,63600.0,11.060369,59672.918,10.996634,3927.082031,214,47295.0,10.764160,42423.773,10.655464,4871.226562,205.0,28,high,0.114823,0.114331,0.065810,0.065537,0.079378,0.078820,7.0,6.0,7.0,-1.0,1.0,0.0,0.577350,0.021383,0.183302,0.078820,161428.058,551.0,0.998326,0.087524,4709.632813,59331.367,0.079378,True
3,100,3,149222,1,Public,23.0,IL,37.714193,-89.217273,32,14.0,0.8688,37454.0,0.655376,2.0,22.0,1,1055.0,24.0,3237.0,open,11.051382,10.829567,"Agriculture, General.",Southern Illinois University-Carbondale,63031.0,50491.836,12539.164062,57596.0,10.961208,53280.395,10.883324,4315.605469,47,39700.0,10.589106,39023.470,10.571919,676.531250,22.0,28,high,0.017337,0.016352,0.080998,0.079023,0.248340,0.235131,12.0,5.0,1.0,-7.0,-4.0,-11.0,5.567764,0.206213,0.183302,0.079023,142795.701,92.0,0.986666,0.090667,4315.605469,50491.836,0.085471,True
4,100,3,149772,1,Public,145.0,IL,40.468086,-90.686899,33,13.0,0.7118,36222.0,0.684009,2.0,21.0,1,NaN,NaN,1807.0,open,10.971709,10.874193,"Agriculture, General.",Western Illinois University,58204.0,52796.130,5407.871094,58333.0,10.973923,53879.047,10.894497,4453.953125,160,48509.0,10.789505,39236.290,10.577357,9272.710938,149.0,28,high,0.236330,0.234860,0.082666,0.082183,0.102429,0.101783,4.0,4.0,4.0,0.0,0.0,0.0,0.000000,0.000000,0.183302,0.101783,145911.467,454.0,0.997908,0.111188,5407.871094,52796.130,0.102429,True


In [71]:
join_cols=["unit_id"]
for col in join_cols:
    driver_df[col] = driver_df[col].astype(str).str.strip()
    xgb_residual_df[col] = xgb_residual_df[col].astype(str).str.strip()


### Target Variable Selection

While a composite scoring metric was developed to rank program-level variability, it was not used as the target for the explanatory model.

Instead, the model uses the average  raw percentage error of years 1, 4, and 5 as the target:

  * pct_error = error / predicted

This decision ensures that the model learns directly from observed over- and underperformance, rather than from a derived metric that incorporates additional adjustments (e.g., sample size penalties and variability scaling).

The composite score remains useful for identifying high-variability groups, but the explanatory model focuses on the underlying performance signal.

In [72]:
# residual_df["combined_pct_error"]=(
#     residual_df[['1_year_error','4_year_error','5_year_error']].sum(axis=1)
#     /
#     residual_df[['1_year_pred','4_year_pred','5_year_pred']].sum(axis=1)
# )

In [ ]:
xgb_residual_df["dollar_error_median"] = xgb_residual_df[
    ["1_year_error", "4_year_error", "5_year_error"]
].median(axis=1)

xgb_residual_df["dollar_pred_median"] = xgb_residual_df[
    ["1_year_pred", "4_year_pred", "5_year_pred"]
].median(axis=1)

xgb_residual_df["row_pct_error"] = (
   xgb_residual_df["dollar_error_median"] / xgb_residual_df["dollar_pred_median"]
)

In [74]:
xgb_residual_df.head()

,code,credential_level,unit_id,distance,school_type,5_yr_working_count,school_state,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students,selectivity_bucket,5_year_earning_log,5_year_pred_log,title,school_name,5_year_earning,5_year_pred,5_year_error,4_year_earning,4_year_earning_log,4_year_pred,4_year_pred_log,4_year_error,4_yr_working_count,1_year_earning,1_year_earning_log,1_year_pred,1_year_pred_log,1_year_error,1_yr_working_count,school_count,confidence,1_year_pct_error,1_year_score,4_year_pct_error,4_year_score,5_year_pct_error,5_year_score,rank_1,rank_4,rank_5,move_1_to_4,move_4_to_5,move_1_to_5,rank_std,rank_std_pct,mean_rank_std_pct,median_score,total_pred,total_count,weight,combined_pct_error,dollar_error_median,dollar_pred_median,row_pct_error,sign_agreement
0,100,3,110422,1,Public,21.0,CA,35.299513,-120.657311,23,16.0,0.3132,74513.0,0.455561,2.0,20.0,1,NaN,NaN,1036.0,mid,11.132397,11.212030,"Agriculture, General.",California Polytechnic State University-San Lu...,68350.0,74015.540,-5665.539062,84412.0,11.343465,72705.680,11.194175,11706.320313,36,64786.0,11.078845,45640.770,10.728557,19145.230469,18.0,28,high,0.419477,0.389775,0.161010,0.155674,-0.076545,-0.072040,1.0,2.0,14.0,1.0,12.0,13.0,7.234178,0.267933,0.183302,0.155674,192361.990,75.0,0.983097,0.182567,11706.320313,72705.680,0.161010,False
1,100,3,130934,1,Public,24.0,DE,39.187173,-75.540530,13,14.0,0.4658,39554.0,0.685488,2.0,20.0,1,940.0,NaN,746.0,mid,10.768022,10.985131,"Agriculture, General.",Delaware State University,47478.0,58990.477,-11512.476562,52676.0,10.871915,58480.465,10.976448,-5804.464844,24,38873.0,10.568055,40523.883,10.609647,-1650.882813,22.0,28,high,-0.040739,-0.038425,-0.099255,-0.094049,-0.195158,-0.185260,18.0,16.0,24.0,-2.0,8.0,6.0,4.163332,0.154197,0.183302,-0.094049,157994.825,70.0,0.981691,-0.110215,-5804.464844,58480.465,-0.099255,True
2,100,3,145813,1,Public,132.0,IL,40.509403,-88.990058,22,16.0,0.8815,67099.0,0.481270,2.0,20.0,1,1113.0,24.0,2439.0,open,11.067279,10.990893,"Agriculture, General.",Illinois State University,64041.0,59331.367,4709.632813,63600.0,11.060369,59672.918,10.996634,3927.082031,214,47295.0,10.764160,42423.773,10.655464,4871.226562,205.0,28,high,0.114823,0.114331,0.065810,0.065537,0.079378,0.078820,7.0,6.0,7.0,-1.0,1.0,0.0,0.577350,0.021383,0.183302,0.078820,161428.058,551.0,0.998326,0.087524,4709.632813,59331.367,0.079378,True
3,100,3,149222,1,Public,23.0,IL,37.714193,-89.217273,32,14.0,0.8688,37454.0,0.655376,2.0,22.0,1,1055.0,24.0,3237.0,open,11.051382,10.829567,"Agriculture, General.",Southern Illinois University-Carbondale,63031.0,50491.836,12539.164062,57596.0,10.961208,53280.395,10.883324,4315.605469,47,39700.0,10.589106,39023.470,10.571919,676.531250,22.0,28,high,0.017337,0.016352,0.080998,0.079023,0.248340,0.235131,12.0,5.0,1.0,-7.0,-4.0,-11.0,5.567764,0.206213,0.183302,0.079023,142795.701,92.0,0.986666,0.090667,4315.605469,50491.836,0.085471,True
4,100,3,149772,1,Public,145.0,IL,40.468086,-90.686899,33,13.0,0.7118,36222.0,0.684009,2.0,21.0,1,NaN,NaN,1807.0,open,10.971709,10.874193,"Agriculture, General.",Western Illinois University,58204.0,52796.130,5407.871094,58333.0,10.973923,53879.047,10.894497,4453.953125,160,48509.0,10.789505,39236.290,10.577357,9272.710938,149.0,28,high,0.236330,0.234860,0.082666,0.082183,0.102429,0.101783,4.0,4.0,4.0,0.0,0.0,0.0,0.000000,0.000000,0.183302,0.101783,145911.467,454.0,0.997908,0.111188,5407.871094,52796.130,0.102429,True


In [75]:
school_df = xgb_residual_df.groupby("unit_id", as_index=False).agg(
    combined_pct_error=("row_pct_error", "median"),
    avg_rank_stability=("mean_rank_std_pct", "mean"),
    school_name=("school_name", "first"),
    total_count_1=("1_yr_working_count", "sum"),
    total_count_4=("4_yr_working_count", "sum"),
    total_count_5=("5_yr_working_count", "sum"),
)

school_df["total_count"] = (
    school_df["total_count_1"] +
    school_df["total_count_4"] +
    school_df["total_count_5"]
)

k = np.percentile(np.log1p(school_df["total_count"]), 75)
school_df["weight"] = (
    np.log1p(school_df["total_count"]) /
    np.log1p(school_df["total_count"] + k)
)

In [76]:
targ="combined_pct_error"
merge_df=driver_df.merge(school_df[[targ,'unit_id','weight']], on=["unit_id"],how="inner")
merge_df.head()

,program_percentage_agriculture,program_percentage_resources,program_percentage_architecture,program_percentage_ethnic_cultural_gender,program_percentage_communication,program_percentage_communications_technology,program_percentage_computer,program_percentage_personal_culinary,program_percentage_education,program_percentage_engineering,program_percentage_engineering_technology,program_percentage_language,program_percentage_family_consumer_science,program_percentage_legal,program_percentage_english,program_percentage_humanities,program_percentage_library,program_percentage_biological,program_percentage_mathematics,program_percentage_military,program_percentage_multidiscipline,program_percentage_parks_recreation_fitness,program_percentage_philosophy_religious,program_percentage_theology_religious_vocation,program_percentage_physical_science,program_percentage_science_technology,program_percentage_psychology,program_percentage_security_law_enforcement,program_percentage_public_administration_social_service,program_percentage_social_science,program_percentage_construction,program_percentage_mechanic_repair_technology,program_percentage_precision_production,program_percentage_transportation,program_percentage_visual_performing,program_percentage_health,program_percentage_business_marketing,program_percentage_history,instructional_expenditure_per_fte,faculty_salary,ft_faculty_rate,program_reporter_programs_offered,student_faculty_ratio,endowment_begin,endowment_end,dolflag,school_name,unit_id,has_endowment,combined_pct_error,weight
0,0.0407,0.0000,0.0136,0.0000,0.0000,0.0542,0.0424,0.0,0.0424,0.1085,0.0203,0.0000,0.0186,0.0,0.0119,0.0661,0.0,0.1424,0.0051,0.0,0.0000,0.0373,0.0000,0.0,0.0237,0.0000,0.0559,0.0644,0.0441,0.0220,0.0,0.0,0.0,0.0,0.0186,0.0000,0.1678,0.0000,7254.0,8699.0,0.6439,NaN,19.0,NaN,NaN,0.0,Alabama A & M University,100654,0,0.004683,0.999317
1,0.0000,0.0000,0.0000,0.0007,0.0189,0.0000,0.0352,0.0,0.0541,0.0541,0.0000,0.0078,0.0000,0.0,0.0150,0.0303,0.0,0.1489,0.0055,0.0,0.0036,0.0000,0.0036,0.0,0.0176,0.0007,0.0762,0.0358,0.0163,0.0267,0.0,0.0,0.0,0.0,0.0274,0.2111,0.1997,0.0108,17855.0,12612.0,0.7704,NaN,18.0,7.393729e+08,8.589892e+08,0.0,University of Alabama at Birmingham,100663,1,0.004397,0.999934
2,0.0000,0.0000,0.0000,0.0000,0.0102,0.0000,0.0752,0.0,0.0223,0.3155,0.0122,0.0034,0.0020,0.0,0.0223,0.0000,0.0,0.0616,0.0156,0.0,0.0156,0.0271,0.0007,0.0,0.0427,0.0000,0.0251,0.0000,0.0000,0.0156,0.0,0.0,0.0,0.0,0.0413,0.1043,0.1774,0.0102,9877.0,10639.0,0.6590,NaN,17.0,9.962702e+07,1.137376e+08,1.0,University of Alabama in Huntsville,100706,1,0.028203,0.999451
3,0.0000,0.0000,0.0000,0.0000,0.0511,0.0000,0.0404,0.0,0.0745,0.0128,0.0000,0.0000,0.0000,0.0,0.0085,0.0000,0.0,0.1085,0.0064,0.0,0.0979,0.0128,0.0000,0.0,0.0213,0.0000,0.0638,0.1234,0.0383,0.0298,0.0,0.0,0.0,0.0,0.1043,0.0872,0.1191,0.0000,10723.0,8153.0,0.6477,NaN,15.0,1.186163e+08,1.351989e+08,0.0,Alabama State University,100724,1,0.040910,0.999299
4,0.0000,0.0068,0.0000,0.0014,0.0950,0.0000,0.0163,0.0,0.0250,0.1002,0.0000,0.0033,0.0628,0.0,0.0098,0.0002,0.0,0.0375,0.0080,0.0,0.0138,0.0392,0.0021,0.0,0.0098,0.0000,0.0517,0.0000,0.0100,0.0900,0.0,0.0,0.0,0.0,0.0241,0.0920,0.2911,0.0100,9728.0,11419.0,0.7904,NaN,19.0,1.369440e+09,1.565892e+09,1.0,The University of Alabama,100751,1,-0.003700,0.999954


In [77]:
model_df=merge_df.copy()
print("scorecard driver schools:", driver_df['unit_id'].nunique())
print("school_df school:", school_df['unit_id'].nunique())
print("model_df schools:", model_df['unit_id'].nunique())

print("model_df columns:")
print(sorted(model_df.columns.tolist()))

scorecard driver schools: 6322
school_df school: 4327
model_df schools: 4327
model_df columns:
['combined_pct_error', 'dolflag', 'endowment_begin', 'endowment_end', 'faculty_salary', 'ft_faculty_rate', 'has_endowment', 'instructional_expenditure_per_fte', 'program_percentage_agriculture', 'program_percentage_architecture', 'program_percentage_biological', 'program_percentage_business_marketing', 'program_percentage_communication', 'program_percentage_communications_technology', 'program_percentage_computer', 'program_percentage_construction', 'program_percentage_education', 'program_percentage_engineering', 'program_percentage_engineering_technology', 'program_percentage_english', 'program_percentage_ethnic_cultural_gender', 'program_percentage_family_consumer_science', 'program_percentage_health', 'program_percentage_history', 'program_percentage_humanities', 'program_percentage_language', 'program_percentage_legal', 'program_percentage_library', 'program_percentage_mathematics', 'pro

In [78]:
model_df=model_df.drop(columns=[
    # "program_reporter_programs_offered",
    # 'code',
    # 'unit_id',
    # 'school_name'
    ]
)

In [79]:
program_cols = [c for c in model_df.columns if c.startswith("program_percentage_")]

for c in program_cols:
    model_df[c] = pd.to_numeric(model_df[c], errors="coerce").fillna(0)

# STEM
model_df["pct_stem"] = model_df[
    [
        "program_percentage_computer",
        "program_percentage_engineering",
        "program_percentage_engineering_technology",
        "program_percentage_mathematics",
        "program_percentage_physical_science",
        "program_percentage_biological",
        "program_percentage_science_technology"
    ]
].sum(axis=1)

# Business / Econ
model_df["pct_business"] = model_df[
    ["program_percentage_business_marketing"]
].sum(axis=1)

# Health
model_df["pct_health"] = model_df[
    ["program_percentage_health"]
].sum(axis=1)

# Social Sciences
model_df["pct_social_science"] = model_df[
    [
        "program_percentage_psychology",
        "program_percentage_social_science",
        "program_percentage_history",
        "program_percentage_public_administration_social_service"
    ]
].sum(axis=1)

# Humanities
model_df["pct_humanities"] = model_df[
    [
        "program_percentage_english",
        "program_percentage_language",
        "program_percentage_humanities",
        "program_percentage_philosophy_religious",
        "program_percentage_theology_religious_vocation",
        "program_percentage_ethnic_cultural_gender"
    ]
].sum(axis=1)

# Arts & Communication
model_df["pct_arts_comm"] = model_df[
    [
        "program_percentage_visual_performing",
        "program_percentage_communication"
    ]
].sum(axis=1)

# Education
model_df["pct_education"] = model_df[
    ["program_percentage_education"]
].sum(axis=1)

# Trades / Technical
model_df["pct_trades"] = model_df[
    [
        "program_percentage_construction",
        "program_percentage_mechanic_repair_technology",
        "program_percentage_precision_production",
        "program_percentage_transportation"
    ]
].sum(axis=1)

# Services / Consumer
model_df["pct_services"] = model_df[
    [
        "program_percentage_personal_culinary",
        "program_percentage_family_consumer_science",
        "program_percentage_parks_recreation_fitness"
    ]
].sum(axis=1)

# Law / Security
model_df["pct_law_security"] = model_df[
    [
        "program_percentage_legal",
        "program_percentage_security_law_enforcement"
    ]
].sum(axis=1)

# Agriculture / Natural resources
model_df["pct_agriculture"] = model_df[
    [
        # "program_percentage_agriculture",
        "program_percentage_resources"
    ]
].sum(axis=1)

model_df["pct_high_roi"] = (
    model_df["program_percentage_engineering"] +
    model_df["program_percentage_computer"] +
    model_df["program_percentage_health"]
)

model_df["pct_low_roi"] = (
    model_df["program_percentage_education"] +
    model_df["program_percentage_personal_culinary"] +
    model_df["program_percentage_humanities"]
)

model_df["program_hhi"] = (model_df[program_cols] ** 2).sum(axis=1)

model_df["max_program_share"] = model_df[program_cols].max(axis=1)

model_df["high_roi_x_concentration"] = (
    model_df["pct_high_roi"] * model_df["program_hhi"]
)

# model_df = model_df.drop(columns=program_cols)



In [80]:
display("Repeated columns within instituions",(model_df.groupby("unit_id").nunique() > 1).sum())

'Repeated columns within instituions'

program_percentage_agriculture               0
program_percentage_resources                 0
program_percentage_architecture              0
program_percentage_ethnic_cultural_gender    0
program_percentage_communication             0
                                            ..
pct_high_roi                                 0
pct_low_roi                                  0
program_hhi                                  0
max_program_share                            0
high_roi_x_concentration                     0
Length: 66, dtype: int64

In [81]:
inst_model_df = model_df.drop_duplicates(subset="unit_id").reset_index(drop=True)
print("inst_model_df shape:", inst_model_df.shape)
display(inst_model_df.head())

inst_model_df shape: (4327, 67)


,program_percentage_agriculture,program_percentage_resources,program_percentage_architecture,program_percentage_ethnic_cultural_gender,program_percentage_communication,program_percentage_communications_technology,program_percentage_computer,program_percentage_personal_culinary,program_percentage_education,program_percentage_engineering,program_percentage_engineering_technology,program_percentage_language,program_percentage_family_consumer_science,program_percentage_legal,program_percentage_english,program_percentage_humanities,program_percentage_library,program_percentage_biological,program_percentage_mathematics,program_percentage_military,program_percentage_multidiscipline,program_percentage_parks_recreation_fitness,program_percentage_philosophy_religious,program_percentage_theology_religious_vocation,program_percentage_physical_science,program_percentage_science_technology,program_percentage_psychology,program_percentage_security_law_enforcement,program_percentage_public_administration_social_service,program_percentage_social_science,program_percentage_construction,program_percentage_mechanic_repair_technology,program_percentage_precision_production,program_percentage_transportation,program_percentage_visual_performing,program_percentage_health,program_percentage_business_marketing,program_percentage_history,instructional_expenditure_per_fte,faculty_salary,ft_faculty_rate,program_reporter_programs_offered,student_faculty_ratio,endowment_begin,endowment_end,dolflag,school_name,unit_id,has_endowment,combined_pct_error,weight,pct_stem,pct_business,pct_health,pct_social_science,pct_humanities,pct_arts_comm,pct_education,pct_trades,pct_services,pct_law_security,pct_agriculture,pct_high_roi,pct_low_roi,program_hhi,max_program_share,high_roi_x_concentration
0,0.0407,0.0000,0.0136,0.0000,0.0000,0.0542,0.0424,0.0,0.0424,0.1085,0.0203,0.0000,0.0186,0.0,0.0119,0.0661,0.0,0.1424,0.0051,0.0,0.0000,0.0373,0.0000,0.0,0.0237,0.0000,0.0559,0.0644,0.0441,0.0220,0.0,0.0,0.0,0.0,0.0186,0.0000,0.1678,0.0000,7254.0,8699.0,0.6439,NaN,19.0,NaN,NaN,0.0,Alabama A & M University,100654,0,0.004683,0.999317,0.3424,0.1678,0.0000,0.1220,0.0780,0.0186,0.0424,0.0,0.0559,0.0644,0.0000,0.1509,0.1085,0.085876,0.1678,0.012959
1,0.0000,0.0000,0.0000,0.0007,0.0189,0.0000,0.0352,0.0,0.0541,0.0541,0.0000,0.0078,0.0000,0.0,0.0150,0.0303,0.0,0.1489,0.0055,0.0,0.0036,0.0000,0.0036,0.0,0.0176,0.0007,0.0762,0.0358,0.0163,0.0267,0.0,0.0,0.0,0.0,0.0274,0.2111,0.1997,0.0108,17855.0,12612.0,0.7704,NaN,18.0,7.393729e+08,8.589892e+08,0.0,University of Alabama at Birmingham,100663,1,0.004397,0.999934,0.2620,0.1997,0.2111,0.1300,0.0574,0.0463,0.0541,0.0,0.0000,0.0358,0.0000,0.3004,0.0844,0.124569,0.2111,0.037421
2,0.0000,0.0000,0.0000,0.0000,0.0102,0.0000,0.0752,0.0,0.0223,0.3155,0.0122,0.0034,0.0020,0.0,0.0223,0.0000,0.0,0.0616,0.0156,0.0,0.0156,0.0271,0.0007,0.0,0.0427,0.0000,0.0251,0.0000,0.0000,0.0156,0.0,0.0,0.0,0.0,0.0413,0.1043,0.1774,0.0102,9877.0,10639.0,0.6590,NaN,17.0,9.962702e+07,1.137376e+08,1.0,University of Alabama in Huntsville,100706,1,0.028203,0.999451,0.5228,0.1774,0.1043,0.0509,0.0264,0.0515,0.0223,0.0,0.0291,0.0000,0.0000,0.4950,0.0223,0.158330,0.3155,0.078373
3,0.0000,0.0000,0.0000,0.0000,0.0511,0.0000,0.0404,0.0,0.0745,0.0128,0.0000,0.0000,0.0000,0.0,0.0085,0.0000,0.0,0.1085,0.0064,0.0,0.0979,0.0128,0.0000,0.0,0.0213,0.0000,0.0638,0.1234,0.0383,0.0298,0.0,0.0,0.0,0.0,0.1043,0.0872,0.1191,0.0000,10723.0,8153.0,0.6477,NaN,15.0,1.186163e+08,1.351989e+08,0.0,Alabama State University,100724,1,0.040910,0.999299,0.1894,0.1191,0.0872,0.1319,0.0085,0.1554,0.0745,0.0,0.0128,0.1234,0.0000,0.1404,0.0745,0.086365,0.1234,0.012126
4,0.0000,0.0068,0.0000,0.0014,0.0950,0.0000,0.0163,0.0,0.0250,0.1002,0.0000,0.0033,0.0628,0.0,0.0098,0.0002,0.0,0.0375,0.0080,0.0,0.0138,0.0392,0.0021,0.0,0.0098,0.0000,0.0517,0.0000,0.0100,0.0900,0.0,0.0,0.0,0.0,0.0241,0.0920,0.2911,0.0100,9728.0,11419.0,0.7904,NaN,19.0,1.369440e+09,1.565892e+09,1.0,The University of Alabama,100751,1,-0.0037

In [ ]:
display(inst_model_df.describe())

numeric_cols = inst_model_df.select_dtypes(include=[np.number]).columns.tolist()
display("Feature correlation to target variable",
        inst_model_df[numeric_cols].corr()[targ].sort_values())
display(inst_model_df.info())

,program_percentage_agriculture,program_percentage_resources,program_percentage_architecture,program_percentage_ethnic_cultural_gender,program_percentage_communication,program_percentage_communications_technology,program_percentage_computer,program_percentage_personal_culinary,program_percentage_education,program_percentage_engineering,program_percentage_engineering_technology,program_percentage_language,program_percentage_family_consumer_science,program_percentage_legal,program_percentage_english,program_percentage_humanities,program_percentage_library,program_percentage_biological,program_percentage_mathematics,program_percentage_military,program_percentage_multidiscipline,program_percentage_parks_recreation_fitness,program_percentage_philosophy_religious,program_percentage_theology_religious_vocation,program_percentage_physical_science,program_percentage_science_technology,program_percentage_psychology,program_percentage_security_law_enforcement,program_percentage_public_administration_social_service,program_percentage_social_science,program_percentage_construction,program_percentage_mechanic_repair_technology,program_percentage_precision_production,program_percentage_transportation,program_percentage_visual_performing,program_percentage_health,program_percentage_business_marketing,program_percentage_history,instructional_expenditure_per_fte,faculty_salary,ft_faculty_rate,program_reporter_programs_offered,student_faculty_ratio,endowment_begin,endowment_end,dolflag,has_endowment,combined_pct_error,weight,pct_stem,pct_business,pct_health,pct_social_science,pct_humanities,pct_arts_comm,pct_education,pct_trades,pct_services,pct_law_security,pct_agriculture,pct_high_roi,pct_low_roi,program_hhi,max_program_share,high_roi_x_concentration
count,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4321.000000,3072.000000,2912.000000,1409.000000,4202.000000,2.251000e+03,2.251000e+03,4267.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000,4327.000000
mean,0.011191,0.004211,0.001552,0.001072,0.012648,0.003737,0.029479,0.181065,0.025790,0.013613,0.014441,0.002416,0.005930,0.002744,0.006132,0.067669,0.000057,0.024271,0.003424,0.000409,0.011485,0.012945,0.003921,0.005529,0.004854,0.000808,0.026518,0.020064,0.007920,0.018383,0.016018,0.032422,0.016612,0.009413,0.027515,0.248758,0.091439,0.003968,9980.008794,8333.480143,0.607719,5.617459,16.342932,3.669611e+08,3.925047e+08,0.434263,0.520222,0.005661,0.996526,0.090889,0.091439,0.248758,0.056788,0.086738,0.040163,0.025790,0.074465,0.199940,0.022807,0.004211,0.291849,0.274524,0.469220,0.554762,0.165711
std,0.058359,0.018008,0.018414,0.006609,0.044013,0.032788,0.060959,0.369368,0.057262,0.055116,0.048717,0.007959,0.021419,0.026339,0.017487,0.142046,0.001136,0.053937,0.009208,0.008323,0.033631,0.033141,0.048856,0.055035,0.017323,0.007763,0.057192,0.045781,0.027099,0.046114,0.072963,0.116686,0.068208,0.064500,0.099337,0.322664,0.125168,0.009037,15447.418759,2838.348431,0.277490,5.893275,7.694457,2.154191e+09,2.249583e+09,0.495718,0.499649,0.106238,0.005290,0.132947,0.125168,0.322664,0.097986,0.158910,0.111651,0.057262,0.192851,0.362832,0.053301,0.018008,0.319469,0.358955,0.377284,0.337549,0.310759
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0

'Feature correlation to target variable'

program_percentage_theology_religious_vocation   -0.071302
program_percentage_agriculture                   -0.059375
pct_humanities                                   -0.054078
program_percentage_philosophy_religious          -0.040880
pct_arts_comm                                    -0.040393
                                                    ...   
program_percentage_engineering                    0.051630
program_percentage_mechanic_repair_technology     0.078997
pct_trades                                        0.081066
faculty_salary                                    0.140828
combined_pct_error                                1.000000
Name: combined_pct_error, Length: 65, dtype: float64

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4327 entries, 0 to 4326
Data columns (total 67 columns):
 #   Column                                                   Non-Null Count  Dtype  
---  ------                                                   --------------  -----  
 0   program_percentage_agriculture                           4327 non-null   float64
 1   program_percentage_resources                             4327 non-null   float64
 2   program_percentage_architecture                          4327 non-null   float64
 3   program_percentage_ethnic_cultural_gender                4327 non-null   float64
 4   program_percentage_communication                         4327 non-null   float64
 5   program_percentage_communications_technology             4327 non-null   float64
 6   program_percentage_computer                              4327 non-null   float64
 7   program_percentage_personal_culinary                     4327 non-null   float64
 8   program_percentage_education

None

In [ ]:
def run_school_corr_screen(merge_df, feature_cols, target_col="combined_pct_error", weight_col="weight"):
    
    df_tmp = merge_df[feature_cols + [target_col, weight_col]].copy()
    df_tmp = df_tmp.dropna(subset=[target_col])
    df_tmp = df_tmp.dropna(subset=feature_cols, how="all")
    
    print(f"Rows after dropna: {len(df_tmp)}")
    print(f"Features: {len(feature_cols)}")
    print(f"Target distribution:\n{df_tmp[target_col].describe().round(4)}\n")

    lo, hi = df_tmp[target_col].quantile(0.02), df_tmp[target_col].quantile(0.98)
    df_tmp[target_col] = df_tmp[target_col].clip(lo, hi)

    X = df_tmp[feature_cols].apply(pd.to_numeric, errors="coerce")
    y = df_tmp[target_col]
    w = df_tmp[weight_col]

    corr = X.corrwith(y).abs().sort_values(ascending=False)
    
    print("Top 15 correlations with target:")
    print(corr.head(15).round(4).to_string())
    print(f"\nFeatures with |corr| > 0.10: {(corr > 0.10).sum()}")
    print(f"Features with |corr| > 0.15: {(corr > 0.15).sum()}")
    print(f"Features with |corr| > 0.20: {(corr > 0.20).sum()}")

    from sklearn.ensemble import RandomForestRegressor
    from sklearn.model_selection import KFold
    from sklearn.metrics import r2_score
    from sklearn.impute import SimpleImputer

    imputer = SimpleImputer(strategy="median")
    X_imp = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    rf = RandomForestRegressor(
        n_estimators=100, 
        max_depth=3,
        random_state=42, 
        n_jobs=-1
    )

    fold_r2s = []
    for train_idx, val_idx in kf.split(X_imp):
        X_tr, X_val = X_imp.iloc[train_idx], X_imp.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        w_tr, w_val = w.iloc[train_idx], w.iloc[val_idx]

        rf.fit(X_tr, y_tr, sample_weight=w_tr)
        y_pred = rf.predict(X_val)
        fold_r2s.append(r2_score(y_val, y_pred, sample_weight=w_val))

    print(f"\nRF CV R² (max_depth=3): {np.mean(fold_r2s):.4f} ± {np.std(fold_r2s):.4f}")
    print(f"Fold R²s: {[round(s,3) for s in fold_r2s]}")

    
    if np.mean(fold_r2s) > 0:
        rf.fit(X_imp, y, sample_weight=w)
        imp_df = pd.DataFrame({
            "feature": feature_cols,
            "importance": rf.feature_importances_
        }).sort_values("importance", ascending=False)
        print("\nTop 10 feature importances:")
        print(imp_df.head(10).to_string(index=False))

    return corr

feature_cols = [c for c in inst_model_df.columns if c not in [
    "unit_id", "school_name", "weight", "combined_pct_error",
    "avg_rank_stability", "total_pred", "median_error",
    "total_count", "total_count_1", "total_count_4", "total_count_5"
]]

corr_results = run_school_corr_screen(inst_model_df, feature_cols)

Rows after dropna: 4327
Features: 63
Target distribution:
count    4327.0000
mean        0.0057
std         0.1062
min        -0.5258
25%        -0.0438
50%         0.0017
75%         0.0493
max         1.4367
Name: combined_pct_error, dtype: float64

Top 15 correlations with target:
faculty_salary                                    0.1523
pct_trades                                        0.0864
program_percentage_mechanic_repair_technology     0.0790
program_percentage_theology_religious_vocation    0.0740
pct_arts_comm                                     0.0709
program_percentage_agriculture                    0.0684
program_percentage_visual_performing              0.0646
program_percentage_engineering                    0.0613
pct_humanities                                    0.0497
program_percentage_engineering_technology         0.0462
program_percentage_architecture                   0.0436
pct_stem                                          0.0418
program_percentage_philosophy_r

In [ ]:
stage1_vars = [
    'unit_id',
    'school_state',
    'school_type',
    'locale',
    'carnegie_size_setting',
    'selectivity_bucket',
    'open_admissions_policy',
    'title_iv_eligibility_type',
    'admission_rate_overall',
    'median_family_income',
    'students_with_pell_grant',
    'age_entry',
    'sat_scores_average_overall',
    'act_scores_midpoint_cumulative',
]

school_level_extras = school_df[[
    'unit_id',
    'total_count',
    'avg_rank_stability',  
]]

stage1_meta = (
    xgb_residual_df[stage1_vars]
    .drop_duplicates(subset='unit_id')
    .reset_index(drop=True)
)

print(f"stage1_meta shape: {stage1_meta.shape}")
print(f"inst_model_df schools: {inst_model_df['unit_id'].nunique()}")
print(f"stage1_meta schools: {stage1_meta['unit_id'].nunique()}")

slice_df = inst_model_df \
    .merge(stage1_meta, on='unit_id', how='left') \
    .merge(school_level_extras, on='unit_id', how='left')

print(f"slice_df shape: {slice_df.shape}")
print(f"Matched stage1: {slice_df['school_state'].notna().sum()}")
print(f"Matched school extras: {slice_df['total_count'].notna().sum()}")

stage1_meta shape: (4327, 14)
inst_model_df schools: 4327
stage1_meta schools: 4327
slice_df shape: (4327, 82)
Matched stage1: 4327
Matched school extras: 4327


In [ ]:
school_sign = (
    xgb_residual_df.groupby('unit_id')['sign_agreement']
    .mean()  # fraction of programs where all 3 years agree
    .reset_index()
    .rename(columns={'sign_agreement': 'school_sign_agreement_rate'})
)

slice_df = slice_df.merge(school_sign, on='unit_id', how='left')

In [ ]:
school_distance = xgb_residual_df.groupby('unit_id').agg(
    pct_fully_online=('distance', lambda x: (x == 3).mean()),
    pct_no_online=('distance', lambda x: (x == 1).mean()),
    pct_hybrid=('distance', lambda x: (x == 2).mean()),
    distance_mode=('distance', lambda x: x.mode()[0])
).reset_index()

slice_df = slice_df.merge(school_distance, on='unit_id', how='left')

In [87]:
slice_df.head()

,program_percentage_agriculture,program_percentage_resources,program_percentage_architecture,program_percentage_ethnic_cultural_gender,program_percentage_communication,program_percentage_communications_technology,program_percentage_computer,program_percentage_personal_culinary,program_percentage_education,program_percentage_engineering,program_percentage_engineering_technology,program_percentage_language,program_percentage_family_consumer_science,program_percentage_legal,program_percentage_english,program_percentage_humanities,program_percentage_library,program_percentage_biological,program_percentage_mathematics,program_percentage_military,program_percentage_multidiscipline,program_percentage_parks_recreation_fitness,program_percentage_philosophy_religious,program_percentage_theology_religious_vocation,program_percentage_physical_science,program_percentage_science_technology,program_percentage_psychology,program_percentage_security_law_enforcement,program_percentage_public_administration_social_service,program_percentage_social_science,program_percentage_construction,program_percentage_mechanic_repair_technology,program_percentage_precision_production,program_percentage_transportation,program_percentage_visual_performing,program_percentage_health,program_percentage_business_marketing,program_percentage_history,instructional_expenditure_per_fte,faculty_salary,ft_faculty_rate,program_reporter_programs_offered,student_faculty_ratio,endowment_begin,endowment_end,dolflag,school_name,unit_id,has_endowment,combined_pct_error,weight,pct_stem,pct_business,pct_health,pct_social_science,pct_humanities,pct_arts_comm,pct_education,pct_trades,pct_services,pct_law_security,pct_agriculture,pct_high_roi,pct_low_roi,program_hhi,max_program_share,high_roi_x_concentration,school_state,school_type,locale,carnegie_size_setting,selectivity_bucket,open_admissions_policy,title_iv_eligibility_type,admission_rate_overall,median_family_income,students_with_pell_grant,age_entry,sat_scores_average_overall,act_scores_midpoint_cumulative,total_count,avg_rank_stability,school_sign_agreement_rate,pct_fully_online,pct_no_online,pct_hybrid,distance_mode
0,0.0407,0.0000,0.0136,0.0000,0.0000,0.0542,0.0424,0.0,0.0424,0.1085,0.0203,0.0000,0.0186,0.0,0.0119,0.0661,0.0,0.1424,0.0051,0.0,0.0000,0.0373,0.0000,0.0,0.0237,0.0000,0.0559,0.0644,0.0441,0.0220,0.0,0.0,0.0,0.0,0.0186,0.0000,0.1678,0.0000,7254.0,8699.0,0.6439,NaN,19.0,NaN,NaN,0.0,Alabama A & M University,100654,0,0.004683,0.999317,0.3424,0.1678,0.0000,0.1220,0.0780,0.0186,0.0424,0.0,0.0559,0.0644,0.0000,0.1509,0.1085,0.085876,0.1678,0.012959,AL,Public,12,14.0,mid,2.0,1,0.5795,23553.0,0.852793,20.0,938.0,18.0,1574.0,0.116255,0.615385,0.000000,0.846154,0.153846,1
1,0.0000,0.0000,0.0000,0.0007,0.0189,0.0000,0.0352,0.0,0.0541,0.0541,0.0000,0.0078,0.0000,0.0,0.0150,0.0303,0.0,0.1489,0.0055,0.0,0.0036,0.0000,0.0036,0.0,0.0176,0.0007,0.0762,0.0358,0.0163,0.0267,0.0,0.0,0.0,0.0,0.0274,0.2111,0.1997,0.0108,17855.0,12612.0,0.7704,NaN,18.0,7.393729e+08,8.589892e+08,0.0,University of Alabama at Birmingham,100663,1,0.004397,0.999934,0.2620,0.1997,0.2111,0.1300,0.0574,0.0463,0.0541,0.0,0.0000,0.0358,0.0000,0.3004,0.0844,0.124569,0.2111,0.037421,AL,Public,12,15.0,open,2.0,1,0.8818,34489.0,0.624930,23.0,1258.0,27.0,12741.0,0.114378,0.458333,0.020833,0.604167,0.375000,1
2,0.0000,0.0000,0.0000,0.0000,0.0102,0.0000,0.0752,0.0,0.0223,0.3155,0.0122,0.0034,0.0020,0.0,0.0223,0.0000,0.0,0.0616,0.0156,0.0,0.0156,0.0271,0.0007,0.0,0.0427,0.0000,0.0251,0.0000,0.0000,0.0156,0.0,0.0,0.0,0.0,0.0413,0.1043,0.1774,0.0102,9877.0,10639.0,0.6590,NaN,17.0,9.962702e+07,1.137376e+08,1.0,University of Alabama in Huntsville,100706,1,0.028203,0.999451,0.5228,0.1774,0.1043,0.0509,0.0264,0.0515,0.0223,0.0,0.0291,0.0000,0.0000,0.4950,0.0223,0.158330,0.3155,0.078373,AL,Public,12,13.0,mid,2.0,1,0.6857,44787.0,0.557137,22.0,1319.0,28.0,1911.0,0.111879,0.400000,0.500000,0.400000,0.100000,3
3,0.0000,0.0000,0.0000,0.0000,0.0511,0.0000,0.0404,0.0,0.0745,0.0128

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV, StratifiedKFold
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PowerTransformer
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score


RANDOM_STATE = 42

In [ ]:
import plotly.express as px


percentiles = [0.1, 0.2, 0.3, 0.7, 0.8, 0.9]
values = np.quantile(slice_df[targ], percentiles)

fig = px.histogram(slice_df[targ], nbins=100, title="Target Distribution with Percentiles")

for p, v in zip(percentiles, values):
    fig.add_vline(
        x=v,
        line_dash="dash",
        annotation_text=f"{int(p*100)}%",
        annotation_position="top"
    )

fig.show()

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.base import clone

targ = "combined_pct_error"

meta_cols = [
    'unit_id', 'school_name', targ, 'weight',
    'avg_rank_stability', 'school_sign_agreement_rate',
    'total_count', 'school_state', 'school_type', 'locale',
    'carnegie_size_setting', 'selectivity_bucket',
    'open_admissions_policy', 'pct_fully_online',
    'pct_no_online', 'pct_hybrid', 'distance_mode', 
    'title_iv_eligibility_type', 'admission_rate_overall', 
    'median_family_income', 'students_with_pell_grant', 'age_entry',
    'sat_scores_average_overall', 'act_scores_midpoint_cumulative',
]

meta_cols = [c for c in meta_cols if c in slice_df.columns]

X_master = slice_df.drop(columns=meta_cols, errors='ignore') \
                   .select_dtypes(include=[np.number])

y_master    = slice_df[targ]
w_master    = slice_df['weight']
meta_master = slice_df[meta_cols] 

print(f"X_master shape: {X_master.shape}")
print(f"Features: {X_master.columns.tolist()}")

(X_train, X_test,
 y_train, y_test,
 w_train, w_test,
 meta_train, meta_test) = train_test_split(
    X_master, y_master, w_master, meta_master,
    test_size=0.2,
    random_state=42
)

for df in [X_train, X_test, y_train, y_test,
           w_train, w_test, meta_train, meta_test]:
    df.reset_index(drop=True, inplace=True)

print(f"\nTrain: {len(X_train)} | Test: {len(X_test)}")
print(f"Test unit_ids: {meta_test['unit_id'].nunique()}")

high = y_master.quantile(0.65)
low  = y_master.quantile(0.30)

clf_mask_train = (y_train <= low) | (y_train >= high)
X_clf_train    = X_train[clf_mask_train].reset_index(drop=True)
y_clf_train    = (y_train[clf_mask_train] >= high).astype(int).reset_index(drop=True)
w_clf_train    = w_train[clf_mask_train].reset_index(drop=True)
meta_clf_train = meta_train[clf_mask_train].reset_index(drop=True)

clf_mask_test  = (y_test <= low) | (y_test >= high)
X_clf_test     = X_test[clf_mask_test].reset_index(drop=True)
y_clf_test     = (y_test[clf_mask_test] >= high).astype(int).reset_index(drop=True)
w_clf_test     = w_test[clf_mask_test].reset_index(drop=True)
meta_clf_test  = meta_test[clf_mask_test].reset_index(drop=True)

print(f"\nReg  train/test: {len(X_train)} / {len(X_test)}")
print(f"Clf  train/test: {len(X_clf_train)} / {len(X_clf_test)}")
print(f"Shared test unit_ids: {len(set(meta_test['unit_id']) & set(meta_clf_test['unit_id']))}")

X_master shape: (4327, 63)
Features: ['program_percentage_agriculture', 'program_percentage_resources', 'program_percentage_architecture', 'program_percentage_ethnic_cultural_gender', 'program_percentage_communication', 'program_percentage_communications_technology', 'program_percentage_computer', 'program_percentage_personal_culinary', 'program_percentage_education', 'program_percentage_engineering', 'program_percentage_engineering_technology', 'program_percentage_language', 'program_percentage_family_consumer_science', 'program_percentage_legal', 'program_percentage_english', 'program_percentage_humanities', 'program_percentage_library', 'program_percentage_biological', 'program_percentage_mathematics', 'program_percentage_military', 'program_percentage_multidiscipline', 'program_percentage_parks_recreation_fitness', 'program_percentage_philosophy_religious', 'program_percentage_theology_religious_vocation', 'program_percentage_physical_science', 'program_percentage_science_technolog

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor, XGBClassifier
from sklearn.metrics import roc_auc_score, r2_score

preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

pipe_reg = Pipeline([
    ("preprocessor", preprocessor),
    ("reg", XGBRegressor(
        learning_rate=0.05,
        max_depth=5,
        n_estimators=200,
        subsample=0.6,
        random_state=42,
        n_jobs=-1,
        verbosity=0
    ))
])

pipe_reg.fit(X_train, y_train, reg__sample_weight=w_train)

reg_test_pred  = pipe_reg.predict(X_test)
print(f"Reg Test R²:  {r2_score(y_test, reg_test_pred):.4f}")

pipe_clf = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", HistGradientBoostingClassifier(
        learning_rate=0.056,
        max_depth=6,
        max_iter=322,
        min_samples_leaf=10,
        l2_regularization=0.276,
        random_state=42
    ))
])

pipe_clf.fit(X_clf_train, y_clf_train, clf__sample_weight=w_clf_train)

clf_test_proba = pipe_clf.predict_proba(X_clf_test)[:, 1]
print(f"Clf Test AUC: {roc_auc_score(y_clf_test, clf_test_proba):.4f}")

eval_reg = meta_test.copy()
eval_reg["y_true_reg"]  = y_test
eval_reg["y_pred_reg"]  = reg_test_pred

eval_clf = meta_clf_test.copy()
eval_clf["y_true_clf"]  = y_clf_test
eval_clf["y_proba_clf"] = clf_test_proba
eval_clf["y_pred_clf"]  = (clf_test_proba >= 0.4876).astype(int)

eval_combined = eval_reg.merge(
    eval_clf[['unit_id', 'y_true_clf', 'y_proba_clf', 'y_pred_clf']],
    on='unit_id',
    how='inner'
)

print(f"\nCombined eval shape: {eval_combined.shape}")
print(f"Schools: {len(eval_combined)}")

reg_min = eval_combined["y_pred_reg"].min()
reg_max = eval_combined["y_pred_reg"].max()
eval_combined["y_pred_reg_scaled"] = (
    (eval_combined["y_pred_reg"] - reg_min) / (reg_max - reg_min)
)

Reg Test R²:  0.0469
Clf Test AUC: 0.6159

Combined eval shape: (572, 29)
Schools: 572


In [ ]:
from itertools import combinations
from sklearn.metrics import r2_score, roc_auc_score

baseline_r2  = r2_score(y_test, reg_test_pred)
baseline_auc = roc_auc_score(y_clf_test, clf_test_proba)

print('XGB residuals')
print(f"Baseline Reg R²:  {baseline_r2:.4f}")
print(f"Baseline Clf AUC: {baseline_auc:.4f}")

for df in [eval_reg, eval_clf]:
    df["age_bucket"] = pd.qcut(
        df["age_entry"], q=2,
        labels=["younger", "older"], duplicates="drop"
    )
    df["stability_bucket"] = pd.qcut(
        df["avg_rank_stability"], q=2,
        labels=["stable", "volatile"], duplicates="drop"
    )
    df["school_sign_agreement_rate_bucket"] = pd.qcut(
        df["school_sign_agreement_rate"], q=2,
        labels=["low_agreement", "high_agreement"], duplicates="drop"
    )
    df["weight_bucket"] = pd.qcut(
        df["weight"], q=2,
        labels=["low_weight", "high_weight"], duplicates="drop"
    )

slice_candidates = [
    'age_bucket',
    'stability_bucket',
    'school_type',
    'selectivity_bucket',
    'pct_fully_online',
    'pct_no_online',
    'pct_hybrid',
    'distance_mode',
    'school_sign_agreement_rate_bucket',
    'weight_bucket',
    'total_count', 
    'school_state', 
    'locale',
    'carnegie_size_setting',
    'open_admissions_policy', 
    'title_iv_eligibility_type',
    'median_family_income',
    'students_with_pell_grant', 
    'sat_scores_average_overall', 
    'act_scores_midpoint_cumulative'
]

def run_reg_slices(eval_df, cols, min_samples=150, baseline=baseline_r2):
    results = []
    for col_combo in combinations(cols, len(cols)) if isinstance(cols, list) else [(cols,)]:
        try:
            for vals, group in eval_df.groupby(list(col_combo), observed=True):
                if len(group) < min_samples:
                    continue
                if not isinstance(vals, tuple):
                    vals = (vals,)
                r2 = r2_score(group["y_true_reg"], group["y_pred_reg"])
                label = " | ".join([f"{c}={v}" for c, v in zip(col_combo, vals)])
                results.append({
                    "slice": label, "n_cols": len(col_combo),
                    "n": len(group), "r2": round(r2, 4),
                    "vs_overall": round(r2 - baseline, 4)
                })
        except Exception:
            pass
    return results

def run_clf_slices(eval_df, cols, min_samples=100, baseline=baseline_auc):
    results = []
    for col_combo in combinations(cols, len(cols)) if isinstance(cols, list) else [(cols,)]:
        try:
            for vals, group in eval_df.groupby(list(col_combo), observed=True):
                if len(group) < min_samples:
                    continue
                if group["y_true_clf"].nunique() < 2:
                    continue
                if not isinstance(vals, tuple):
                    vals = (vals,)
                auc = roc_auc_score(group["y_true_clf"], group["y_proba_clf"])
                label = " | ".join([f"{c}={v}" for c, v in zip(col_combo, vals)])
                results.append({
                    "slice": label, "n_cols": len(col_combo),
                    "n": len(group), "auc": round(auc, 4),
                    "vs_overall": round(auc - baseline, 4)
                })
        except Exception:
            pass
    return results

reg_results = []
clf_results = []

for n in [1, 2, 3]:
    for col_combo in combinations(slice_candidates, n):
        reg_results.extend(run_reg_slices(eval_reg, list(col_combo)))
        clf_results.extend(run_clf_slices(eval_clf, list(col_combo)))

reg_slice_df = pd.DataFrame(reg_results).sort_values("r2", ascending=False)
clf_slice_df = pd.DataFrame(clf_results).sort_values("auc", ascending=False)

for n in [1, 2, 3]:
    print(f"\n{'='*65}")
    print(f"REGRESSION — Best {n}-way slices:")
    print(reg_slice_df[reg_slice_df['n_cols']==n].head(5).to_string(index=False))
    print(f"\nREGRESSION — Worst {n}-way slices:")
    print(reg_slice_df[reg_slice_df['n_cols']==n].tail(3).to_string(index=False))

    print(f"\nCLASSIFICATION — Best {n}-way slices:")
    print(clf_slice_df[clf_slice_df['n_cols']==n].head(5).to_string(index=False))
    print(f"\nCLASSIFICATION — Worst {n}-way slices:")
    print(clf_slice_df[clf_slice_df['n_cols']==n].tail(3).to_string(index=False))

XGB residuals
Baseline Reg R²:  0.0469
Baseline Clf AUC: 0.6159

REGRESSION — Best 1-way slices:
                         slice  n_cols   n     r2  vs_overall
school_type=Private, nonprofit       1 261 0.1485      0.1016
            age_bucket=younger       1 437 0.1293      0.0824
    open_admissions_policy=2.0       1 343 0.1195      0.0726
       selectivity_bucket=open       1 211 0.0894      0.0425
       stability_bucket=stable       1 433 0.0868      0.0399

REGRESSION — Worst 1-way slices:
                                          slice  n_cols   n      r2  vs_overall
school_sign_agreement_rate_bucket=low_agreement       1 448 -0.0634     -0.1104
                  carnegie_size_setting=Missing       1 266 -0.0675     -0.1144
                             school_type=Public       1 302 -0.0687     -0.1156

CLASSIFICATION — Best 1-way slices:
                         slice  n_cols   n    auc  vs_overall
       selectivity_bucket=open       1 130 0.7095      0.0936
school_type=Priv

In [ ]:
import itertools
from scipy.optimize import minimize_scalar


def safe_auc(y_true, y_score):
    y_true = pd.Series(y_true)
    if y_true.nunique() < 2:
        return np.nan
    return roc_auc_score(y_true, y_score)


def safe_r2(y_true, y_pred):
    y_true = pd.Series(y_true)
    if len(y_true) < 2:
        return np.nan
    return r2_score(y_true, y_pred)


def find_best_blend_weight(df, reg_col="y_pred_reg_scaled", clf_col="y_proba_clf", y_col="y_true_clf"):
    """
    Find w in [0,1] that maximizes AUC for:
        blended = w * reg_col + (1-w) * clf_col
    """
    if df[y_col].nunique() < 2:
        return np.nan, np.nan

    def neg_auc(w):
        blended = w * df[reg_col] + (1 - w) * df[clf_col]
        return -roc_auc_score(df[y_col], blended)

    res = minimize_scalar(neg_auc, bounds=(0, 1), method="bounded")
    best_w = res.x
    best_auc = -res.fun
    return best_w, best_auc


def evaluate_subset(df, subset_name, min_n=30):
    """
    Evaluate reg-only, clf-only, and blended performance on a subset.
    """
    n = len(df)
    if n < min_n:
        return None

    if df["y_true_clf"].nunique() < 2:
        return None

    reg_r2 = safe_r2(df["y_true_reg"], df["y_pred_reg"])
    clf_auc = safe_auc(df["y_true_clf"], df["y_proba_clf"])
    blend_w, blend_auc = find_best_blend_weight(df)

    if np.isnan(blend_auc):
        return None

    return {
        "slice": subset_name,
        "n": n,
        "reg_r2": reg_r2,
        "clf_auc": clf_auc,
        "blend_w_reg": blend_w,
        "blend_w_clf": 1 - blend_w,
        "blend_auc": blend_auc,
        "blend_gain_vs_clf": blend_auc - clf_auc,
    }


eval_combined = eval_combined.copy()

eval_combined["age_bucket"] = pd.qcut(
    eval_combined["age_entry"],
    q=2,
    labels=["younger", "older"],
    duplicates="drop"
)

eval_combined["stability_bucket"] = pd.qcut(
    eval_combined["avg_rank_stability"],
    q=2,
    labels=["stable", "volatile"],
    duplicates="drop"
)

global_reg_r2 = safe_r2(eval_combined["y_true_reg"], eval_combined["y_pred_reg"])
global_clf_auc = safe_auc(eval_combined["y_true_clf"], eval_combined["y_proba_clf"])
global_w, global_blend_auc = find_best_blend_weight(eval_combined)

print(f"Global reg-only  R² : {global_reg_r2:.4f}")
print(f"Global clf-only  AUC: {global_clf_auc:.4f}")
print(f"Global blend     AUC: {global_blend_auc:.4f}  "
      f"(w_reg={global_w:.3f}, w_clf={1-global_w:.3f})")


segment_cols = [
    "age_bucket",
    "stability_bucket",
    "school_type",
    # Add more if you want:
    # "locale",
    # "open_admissions_policy",
    # "title_iv_eligibility_type",
]

min_n = 75
max_way = 2

results = []

global_row = evaluate_subset(eval_combined, "GLOBAL", min_n=1)
if global_row is not None:
    results.append(global_row)

for k in range(1, max_way + 1):
    for cols in itertools.combinations(segment_cols, k):
        grouped = eval_combined.groupby(list(cols), dropna=False, observed=False)

        for keys, subdf in grouped:
            if k == 1:
                keys = (keys,)

            slice_name = " | ".join(
                f"{col}={val}" for col, val in zip(cols, keys)
            )

            row = evaluate_subset(subdf, slice_name, min_n=min_n)
            if row is not None:
                row["n_cols"] = k
                results.append(row)

results_df = pd.DataFrame(results)

# Optional sort
results_df = results_df.sort_values(
    ["blend_gain_vs_clf", "blend_auc", "n"],
    ascending=[False, False, False]
).reset_index(drop=True)

show_cols = [
    "slice", "n_cols", "n",
    "reg_r2", "clf_auc",
    "blend_w_reg", "blend_w_clf",
    "blend_auc", "blend_gain_vs_clf"
]

print("\nBEST slices for ensemble gain")
display(results_df[show_cols].head(20))

print("\nWORST slices for ensemble gain")
display(results_df[show_cols].sort_values("blend_gain_vs_clf").head(20))

Global reg-only  R² : 0.0688
Global clf-only  AUC: 0.6159
Global blend     AUC: 0.6308  (w_reg=0.724, w_clf=0.276)

BEST slices for ensemble gain


,slice,n_cols,n,reg_r2,clf_auc,blend_w_reg,blend_w_clf,blend_auc,blend_gain_vs_clf
0,"stability_bucket=stable | school_type=Private,...",2.0,84,0.090026,0.649886,0.805246,0.194754,0.714531,0.064645
1,"age_bucket=younger | school_type=Private, nonp...",2.0,117,0.258450,0.693842,0.807351,0.192649,0.741056,0.047214
2,"stability_bucket=stable | school_type=Private,...",2.0,107,0.212041,0.688070,0.699181,0.300819,0.732281,0.044211
3,age_bucket=older | stability_bucket=stable,2.0,123,0.082578,0.619312,0.801187,0.198813,0.660582,0.041270
4,age_bucket=younger | stability_bucket=stable,2.0,153,0.167443,0.689360,0.883005,0.116995,0.729046,0.039685
5,"stability_bucket=('stable',)",1.0,286,0.130354,0.654426,0.943542,0.056458,0.692078,0.037652
6,"age_bucket=older | school_type=Private, for-pr...",2.0,139,-0.017390,0.557167,0.797188,0.202812,0.589533,0.032367
7,"school_type=('Private, nonprofit',)",1.0,176,0.167802,0.690115,0.794038,0.205962,0.717839,0.027724
8,"age_bucket=('older',)",1.0,232,-0.018429,0.565129,0.869861,0.130139,0.589844,0.024715
9,"school_type=('Private, for-profit',)",1.0,217,0.018967,0.523889,0.740325,0.259675,0.545478,0.021589



WORST slices for ensemble gain


,slice,n_cols,n,reg_r2,clf_auc,blend_w_reg,blend_w_clf,blend_auc,blend_gain_vs_clf
20,age_bucket=younger | stability_bucket=volatile,2.0,171,0.048400,0.614129,0.378903,0.621097,0.611934,-0.002195
19,stability_bucket=stable | school_type=Public,2.0,95,-0.003859,0.647585,0.006141,0.993859,0.647585,0.000000
18,"school_type=('Public',)",1.0,179,-0.003704,0.634091,0.069547,0.930453,0.634470,0.000379
17,stability_bucket=volatile | school_type=Public,2.0,84,-0.003461,0.627947,0.042329,0.957671,0.629097,0.001150
16,"stability_bucket=('volatile',)",1.0,286,-0.032285,0.570824,0.386514,0.613486,0.572660,0.001836
15,stability_bucket=volatile | school_type=Privat...,2.0,133,-0.099931,0.450113,0.300969,0.699031,0.453620,0.003507
14,age_bucket=younger | school_type=Public,2.0,132,0.118283,0.702995,0.466233,0.533767,0.709677,0.006682
13,age_bucket=older | stability_bucket=volatile,2.0,109,-0.126687,0.474825,0.381970,0.618030,0.481818,0.006993
12,"age_bucket=('younger',)",1.0,324,0.133259,0.653201,0.723089,0.276911,0.667492,0.014291
11,GLOBAL,NaN,572,0.068839,0.615875,0.724429,0.275571,0.630790,0.014915


## Ridge vs XGB Residuals (Stage 1 to Stage 2 Behavior)

### Key Idea
Stage 1 does not just improve prediction — it **changes the problem that Stage 2 is solving**.

---

### Ridge Residuals
- Residual = what a **simple linear model** could not explain
- Still contains:
  - nonlinear effects
  - interactions
  - structured patterns

Result:
- Stage 2 models perform **stronger**
- Easier to extract **interpretable insights**
- Residual reflects **“deviation from a simple baseline”**

---

### XGBoost Residuals
- Residual = what a **strong nonlinear model** could not explain
- Contains:
  - harder, niche patterns
  - more noise
  - less broad structure

Result:
- Stage 2 models perform **weaker overall**
- Insights are **more localized / niche**
- Residual reflects **“true unexplained remainder”**

---

### Why Stage 2 Looks Worse with XGB
This is expected.

> A stronger Stage 1 removes more predictable signal, leaving a harder problem for Stage 2.

So:
- Lower Stage 2 R²/AUC ≠ worse pipeline
- It means Stage 1 already captured more of the structure

---

### Important Clarification
Differences in Stage 1 performance (e.g., R² or R) do **“cancel out”** numerically in Stage 2.

- Metrics are not additive

Total explainable variance = ```Stage 1 + (Stage 1 residual * Stage 2)```
```
Ridge:  0.76 + (0.24 × 0.26) ≈ 0.76 + 0.0624 = 0.8224
XGB:    0.83 + (0.17 × 0.04) ≈ 0.83 + 0.0085 = 0.8385
```
---

### Practical Interpretation

| Goal | Use |
|------|-----|
| Explainable “expected vs actual” | Ridge residuals |
| Pure leftover / strict baseline | XGB residuals |

---

### Final Takeaway

> Ridge leaves behind **structured, explainable signal**  
> XGB leaves behind **smaller, harder-to-learn signal**

Choose based on whether you want:
- **interpretability (Ridge)**  
- or **clean residual definition (XGB)**

Ridge stage 1 leaves a richer residual that includes both school performance effects and nonlinear constraint effects our model didn't capture. XGB stage 1 produces a purer but harder-to-explain residual. We chose Ridge for stage 1 not because it predicts 
earnings better, XGBoost does that by 7 points,but because it produces a residual that is interpretable and explainable by school-level 
decisions. The two-stage pipeline's total explained variance is comparable (82.2% vs 83.9%), but only the Ridge pipeline yields actionable insights about what schools can influence.